## Lab 2. ML on big data (Apache Spark, SparkML)
## Chast 2. Binarnaya klassifikatsiya: byli li chaevye (high_tip) NYC Taxi

#### Zapusk Spark-sessii

In [ ]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, MinMaxScaler
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, CrossValidatorModel, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

In [ ]:
def create_spark_configuration() -> SparkConf:
    user_name = os.getenv("USER")
    conf = SparkConf()
    conf.setAppName("sobd lab2")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    return conf

In [ ]:
conf = create_spark_configuration()

In [ ]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

#### VARIANT: postav svoy nomer v gruppe mod 4. Model klassifikatsii podstavitsya avtomaticheski.
0 -> RandomForestClassifier | 1 -> GBTClassifier | 2,3 -> LogisticRegression

In [ ]:
VARIANT = 0  # <<< POSTAV SVOY VARIANT (0,1,2,3)

def is_logistic(v): return v in (2, 3)

#### Zagruzka datataseta

In [ ]:
database_name = "shahulov_database"
table_name = "sobd_lab1_processed_table"
spark.catalog.setCurrentDatabase(database_name)
df = spark.table(table_name)
df.show()

#### Postanovka zadachi
Binarnaya tselevaya: high_tip = (tip_amount > 0). Metriki: Precision, Recall, ROC-AUC.
Isklyuchaem tip_amount i total_amount (utechka).

In [ ]:
label_col = "high_tip"
df = df.withColumn(label_col, (F.col("tip_amount") > 0).cast(IntegerType()))
categorical_features = ["PULocationID", "DOLocationID", "RatecodeID"]
numeric_features = ["trip_distance", "fare_amount", "tolls_amount", "trip_duration_min", "passenger_count"]
for col_name in numeric_features:
    df = df.withColumn(col_name, F.col(col_name).cast(DoubleType()))

Stratifitsirovannoe razdelenie + oversampling dlya balansa klassov.

In [ ]:
def stratified_split(data, label, ratio=0.8):
    tr_p, te_p = data.filter(F.col(label) == 1).randomSplit([ratio, 1 - ratio])
    tr_n, te_n = data.filter(F.col(label) == 0).randomSplit([ratio, 1 - ratio])
    return tr_p.union(tr_n), te_p.union(te_n)

def oversample(data, label):
    pos = data.filter(F.col(label) == 1)
    neg = data.filter(F.col(label) == 0)
    tp, tn = pos.count(), neg.count()
    if tp == 0 or tp >= tn:
        return data
    over = pos.withColumn("d", F.explode(F.array_repeat(F.lit(1), tn // tp + 1))).drop("d")
    return neg.union(over)

train_df, test_df = stratified_split(df, label_col, 0.8)
train_df = oversample(train_df, label_col).cache()
test_df = test_df.cache()
train_df.groupBy(label_col).count().show()

#### Konveyer SparkML (model po variantu)

In [ ]:
def get_classifier(v, label):
    if v == 0:
        return RandomForestClassifier(featuresCol="features", labelCol=label, predictionCol="prediction")
    if v == 1:
        return GBTClassifier(featuresCol="features", labelCol=label, predictionCol="prediction", maxIter=20)
    return LogisticRegression(featuresCol="features", labelCol=label, predictionCol="prediction", maxIter=30)

def create_pipeline():
    idx = [f"{f}_idx" for f in categorical_features]
    si = StringIndexer(inputCols=categorical_features, outputCols=idx, handleInvalid="keep")
    if is_logistic(VARIANT):
        ohe_cols = [f"{f}_ohe" for f in categorical_features]
        ohe = OneHotEncoder(inputCols=idx, outputCols=ohe_cols, handleInvalid="keep")
        num_asm = VectorAssembler(inputCols=numeric_features, outputCol="num_vec")
        scaler = MinMaxScaler(inputCol="num_vec", outputCol="num_scaled")
        asm = VectorAssembler(inputCols=ohe_cols + ["num_scaled"], outputCol="features")
        stages = [si, ohe, num_asm, scaler, asm]
    else:
        asm = VectorAssembler(inputCols=idx + numeric_features, outputCol="features")
        stages = [si, asm]
    stages.append(get_classifier(VARIANT, label_col))
    return Pipeline(stages=stages)

pipeline = create_pipeline()

#### Podbor giperparametrov (kross-validatsiya)

In [ ]:
last = pipeline.getStages()[-1]
if VARIANT == 0:
    grid = ParamGridBuilder().addGrid(last.numTrees, [20, 40]).addGrid(last.maxDepth, [5, 7]).build()
elif VARIANT == 1:
    grid = ParamGridBuilder().addGrid(last.maxDepth, [3, 5]).addGrid(last.maxIter, [20]).build()
else:
    grid = ParamGridBuilder().addGrid(last.regParam, [0.01, 0.1]).addGrid(last.elasticNetParam, [0.0, 0.5]).build()

evaluator = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC")
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=grid, evaluator=evaluator, numFolds=3)
cv_model = cv.fit(train_df)

#### Otsenka kachestva (Precision, Recall, ROC-AUC)

In [ ]:
pred = cv_model.transform(test_df)
auc = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC").evaluate(pred)
precision = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol="prediction", metricName="weightedPrecision").evaluate(pred)
recall = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol="prediction", metricName="weightedRecall").evaluate(pred)
f1 = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol="prediction", metricName="f1").evaluate(pred)
print(f"ROC-AUC: {auc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")

#### Sohranenie modeli v HDFS (dlya sleduyushchey laby)

In [ ]:
user_name = os.getenv("USER")
model_path = f"hdfs:///user/{user_name}/models/sobd_lab2_model"
cv_model.bestModel.write().overwrite().save(model_path)
print(f"Model saved: {model_path}")

#### Vyvody
Opishi kachestvo klassifikatora (Precision/Recall/ROC-AUC), balansirovku klassov i giperparametry.